# S6E10 Feature Ladder: What Helps LightGBM?

Which established features help one LightGBM on a shared 5-fold split? Route profiles and original-data signals show
the clearest gains. The default CPU run writes id-keyed OOF/test predictions and scores **public LB 0.96098**.

| step (LightGBM, shared `StratifiedKFold(5, shuffle=True, random_state=42)`) | OOF AUC | ΔAUC | folds Δ>0 |
|---|---:|---:|---:|
| raw 21 columns | 0.959089 | — | — |
| + a route profile | 0.960333 | **+0.001244** | 5/5 |
| + d original lookup / teacher (after a, b, c) | 0.960941 | **+0.000478** | 5/5 |
| final 109-feature set (a b c d f-ev) | 0.961174 | +0.002085 vs raw | — |

| run · model | OOF AUC | public LB |
|---|---:|---:|
| **this notebook, Kaggle CPU, `AUX_MODE='orig'` · LightGBM (`submission.csv`)** | 0.961103 | **0.96098** |
| this notebook, Kaggle CPU, `AUX_MODE='orig'` · XGBoost (`submission_xgb.csv`) | 0.961150 | not reported |
| repository run, competition-data aux · LightGBM | 0.961198 | 0.96101 |
| repository run, competition-data aux · XGBoost | 0.961197 | 0.96102 |

The default Kaggle CPU run produced LightGBM OOF AUC 0.961103 and public LB 0.96098.
Repository runs using competition-data auxiliary features scored LB 0.96101 (LightGBM) and 0.96102 (XGBoost).
No public LB is reported here for the notebook's CPU XGBoost output. All OOF values are *fixed-round refit OOF*
(section 5): not an independent estimate. **Outputs** ([Output tab](https://www.kaggle.com/code/koumeimaki/s6e10-feature-ladder-what-helps-lightgbm/output)): `submission.csv`,
`submission_xgb.csv`, `oof_feature_ladder.csv`, `test_feature_ladder.csv` (id-keyed, with `fold`), `ladder_results.csv`,
`run_metadata.json`. The recorded CPU run completed the notebook computation in 44.5 minutes: approximately 9.6 minutes for auxiliary features, 13.1 for LightGBM, and 20.9 for XGBoost. Runtime varies by session.


## 1. Setup and data

The recorded successful run used CPU and `AUX_MODE='orig'`. The GPU/full-aux path has not been validated in the recorded
Kaggle run. Select the auxiliary mode explicitly (`FL_AUX=orig|full`) and check the resolved input paths and schema
before training (the cell below prints both, plus the library versions).

In [ ]:
import os, sys, time, json, hashlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd, sklearn
import matplotlib, matplotlib.pyplot as plt
import lightgbm as lgb, xgboost as xgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
T_START = time.time()

DEBUG = os.environ.get("FL_DEBUG", "0") == "1"   # subsample + few rounds, for a quick smoke test
RUN_XGB = True                                    # second single model
RUN_PASS1 = False                                 # True: re-derive the round counts with an early-stopped pass
LGB_ROUNDS, XGB_ROUNDS = 1084, 1438               # mean best iteration of an early-stopped pass on these same folds
AUX_ROUNDS = 300
AUX_MODE = os.environ.get("FL_AUX", "orig").strip().lower()   # "orig" (default, recorded run) | "full" (not validated on Kaggle)
if AUX_MODE not in ("orig", "full"):
    raise ValueError(f"FL_AUX must be 'orig' or 'full', got {AUX_MODE!r}")
if DEBUG:
    LGB_ROUNDS, XGB_ROUNDS, AUX_ROUNDS = 150, 150, 20

def has_gpu():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except OSError:
        return False
GPU = has_gpu()
NCPU = os.cpu_count()
if AUX_MODE == "full" and not GPU:
    print("WARNING: AUX_MODE='full' without a GPU fits 65 multi-class models on ~1M rows (hours on 4 CPU cores)")

def first_existing(kaggle_paths, local_rel):
    """Explicit Kaggle mount points first; locally, walk up from the working directory looking for local_rel."""
    for p in kaggle_paths:
        if Path(p).exists():
            return p
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / local_rel).exists():
            return str(d / local_rel)
    raise FileNotFoundError(f"none of {kaggle_paths} or */{local_rel} exists")

COMP = ["/kaggle/input/playground-series-s6e10", "/kaggle/input/competitions/playground-series-s6e10"]
ORIG_PATHS = ["/kaggle/input/binary-aviation-satisfaction-129k/data.csv",
              "/kaggle/input/datasets/arseniyshutko/binary-aviation-satisfaction-129k/data.csv"]
TRAIN = first_existing([f"{c}/train.csv" for c in COMP], "data/train.csv")
TEST = first_existing([f"{c}/test.csv" for c in COMP], "data/test.csv")
ORIG = first_existing(ORIG_PATHS, "data/orig/data.csv")
print("inputs:", TRAIN, TEST, ORIG, sep="\n  ")
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | "
      f"lightgbm {lgb.__version__} | xgboost {xgb.__version__} | matplotlib {matplotlib.__version__}")
print(f"GPU: {GPU} | CPUs: {NCPU} | DEBUG: {DEBUG} | AUX_MODE: {AUX_MODE}")

TARGET, ID, FD = "satisfaction", "id", "Flight Distance"
RATINGS = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking", "Gate location",
           "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment", "On-board service",
           "Leg room service", "Baggage handling", "Checkin service", "Cleanliness"]
NUMERIC = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
CATEGORICAL = ["Gender", "Customer Type", "Type of Travel", "Class"]
FEATURES = NUMERIC[:2] + RATINGS + NUMERIC[2:] + CATEGORICAL
LEVELS = {"Gender": ["Female", "Male"], "Customer Type": ["Loyal Customer", "disloyal Customer"],
          "Type of Travel": ["Business travel", "Personal Travel"], "Class": ["Business", "Eco", "Eco Plus"]}

def labels(s):
    s = pd.Series(s).astype(str).str.lower()
    assert s.isin(["true", "false"]).all()
    return (s == "true").to_numpy().astype(np.int8)

def encode(df):
    out = df[FEATURES].copy()
    for c in CATEGORICAL:
        codes = pd.Categorical(out[c], categories=LEVELS[c]).codes
        assert (codes >= 0).all(), c
        out[c] = codes.astype(np.int8)
    return out

def short(c):
    return c.lower().replace("/", "_").replace(" ", "_").replace("-", "_")

train, test, orig = pd.read_csv(TRAIN), pd.read_csv(TEST), pd.read_csv(ORIG)
# schema checks: fail early if an input is not the expected file
assert set(train.columns) == {ID, TARGET, *FEATURES}, f"train columns: {sorted(train.columns)}"
assert set(test.columns) == {ID, *FEATURES}, f"test columns: {sorted(test.columns)}"
assert set(FEATURES + [TARGET]) <= set(orig.columns), f"original columns: {sorted(orig.columns)}"
assert len(orig) == 129_880, f"expected the 129,880-row original dataset, got {len(orig)} rows"
assert train[ID].is_unique and test[ID].is_unique
if DEBUG:
    train, test = train.sample(60000, random_state=0).reset_index(drop=True), test.iloc[:20000].copy()
y = labels(train[TARGET])
n_train = len(train)
both = encode(pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True))
X_o, y_o = encode(orig), labels(orig[TARGET])
FOLDS = np.full(n_train, -1, np.int8)
for f, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(n_train), y)):
    FOLDS[va] = f
FOLD_HASH = hashlib.sha256(np.stack([train[ID].to_numpy(np.int64), FOLDS.astype(np.int64)]).tobytes()).hexdigest()[:16]
print(f"train {train.shape}  test {test.shape}  original {orig.shape}  positive rate {y.mean():.4f}  id/fold hash {FOLD_HASH}")

## 2. Two facts that drive the features

**Flight Distance is useful as a repeated-value key.** We do not observe actual route identifiers, but the satisfaction
rate depends on *which* distance value a row has, not only on how long the flight is. Below: the rate of every distance
value with ≥ 300 training rows, against a smooth trend; the trend averages up to 10 retained distance values on either
side. If distance only mattered through its length, the dots would scatter around the trend with binomial noise only;
the observed spread is much larger. That is why a label-free **route profile** (per-value means of all other columns)
and per-value target encoding both work.

In [ ]:
tr = pd.DataFrame({"fd": both[FD].iloc[:n_train].to_numpy(), "y": y})
g = tr.groupby("fd")["y"].agg(["mean", "size"]).reset_index()
g = g[g["size"] >= (300 if not DEBUG else 30)].reset_index(drop=True)
trend = g["mean"].rolling(21, center=True, min_periods=5).mean()
resid = g["mean"] - trend
binom_sd = np.sqrt(g["mean"] * (1 - g["mean"]) / g["size"])
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.scatter(g["fd"], g["mean"], s=10, color="#2a78d6", alpha=0.55, linewidths=0, label="one Flight Distance value")
ax.plot(g["fd"], trend, color="#eb6834", lw=2, label="trend: mean of up to 10 retained values on either side")
ax.set_xlabel("Flight Distance (values with ≥ 300 train rows)"); ax.set_ylabel("satisfaction rate")
ax.set_title(f"Same length, different rate: residual SD {resid.std():.3f} vs binomial noise {binom_sd.mean():.3f}", loc="left")
ax.legend(frameon=False, loc="upper left"); ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"{both[FD].nunique()} distinct distances; {len(g)} with enough rows; "
      f"{np.isin(both[FD].iloc[n_train:], both[FD].iloc[:n_train]).mean():.2%} of test distances also occur in train")

**Rating 0 is not simply "worst".** The original data card defines 0 as 'Not Applicable' for inflight wifi service.
The satisfaction rate is not monotonic in the rating: for wifi, 0 has the *highest* rate of all. Trees handle this on
their own. Here, rating values are used directly in route means and auxiliary expected values.

In [ ]:
show = ["Inflight wifi service", "Ease of Online booking", "Online boarding", "Departure/Arrival time convenient"]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
trr = both.iloc[:n_train]
for ax, col in zip(axes, show):
    r = pd.Series(y).groupby(trr[col].to_numpy()).agg(["mean", "size"])
    ax.bar(r.index, r["mean"], color=["#eb6834" if v == 0 else "#2a78d6" for v in r.index], width=0.75)
    for v, m, s in zip(r.index, r["mean"], r["size"]):
        ax.text(v, m + 0.02, f"{m:.2f}", ha="center", fontsize=8)
    ax.set_title(col, fontsize=10, loc="left"); ax.set_xticks(range(6)); ax.set_ylim(0, 1.08)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("satisfaction rate"); fig.suptitle("Satisfaction rate by rating value (orange = 0)", x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 3. The ladder, per fold

Cumulative ladder (each step vs the previous one, LightGBM):

| step | features | OOF AUC | ΔAUC vs previous step | per-fold Δ (×1e-5) | folds Δ>0 |
|---|---:|---:|---:|---|---:|
| raw 21 columns | 21 | 0.959089 | — | — | — |
| + a route profile | 62 | 0.960333 | **+0.001244** | +118 +150 +129 +114 +112 | 5/5 |
| + b value counts | 66 | 0.960402 | **+0.000069** | +4 +13 -2 +17 -2 | 3/5 |
| + c fold-safe TE | 74 | 0.960464 | **+0.000061** | +21 +1 +5 -1 +11 | 4/5 |
| + d original lookup / teacher | 96 | 0.960941 | **+0.000478** | +37 +40 +61 +58 +46 | 5/5 |
| + e rating x context crosses | 135 | 0.960950 | **+0.000009** | -12 +0 +3 +5 +9 | 4/5 |
| + f aux-task (33 cols) | 168 | 0.961089 | **+0.000139** | +29 +13 +4 +10 +7 | 5/5 |
| f -> expected values only (13) | 148 | 0.961114 | **+0.000025** | +1 -2 +17 -4 +8 | 3/5 |
| - e  (final set) | 109 | 0.961174 | **+0.000060** | -0 +12 -2 +12 +2 | 3/5 |

Per-fold deltas are rounded to 1e-5, so `+0`/`-0` mean |Δ| < 0.5e-5; *folds Δ>0* counts the unrounded values. ΔAUC is computed from unrounded AUCs and can differ from the difference of the displayed AUCs by 1e-6.

* Raw → final: **0.959089 → 0.961174 (+0.002085)** OOF AUC with LightGBM.
  The two big steps are **route profile (+0.001244, 5/5 folds)** and
  **original-data lookup + teacher (+0.000478, 5/5)**.
* Adding the 33-column auxiliary block improved pooled OOF AUC by 0.000139. Keeping only the 13 rating
  expectations changed AUC by +0.000025.
* In this LightGBM ablation, rating-context crosses reduced AUC when added to raw features
  (-0.000479, 0/5 folds positive). Their incremental effect in the full configuration was small
  and mixed across folds (+0.000009 when added; removing them +0.000060, 3/5 positive), so we
  omitted them from the final feature set. The displayed ladder does not establish the same effect for XGBoost.
* Fold-safe TE looks small on top of the route profile (+0.000061), but removing it from the full set costs
  -0.000296 (0/5 folds positive): the two blocks overlap but are not substitutes.

The figure splits the same rows by effect size so that the small steps are not flattened by the route-profile gain.
Bars are pooled paired ΔAUC; dots are the five per-fold deltas. *cum* = cumulative step, *alone* = one block on top of
the raw columns, *drop* = one block removed from the full set (a b c d e f-ev, OOF 0.961114).

One block alone on top of raw, and one block removed from the full set:

| step | features | OOF AUC | ΔAUC vs raw | per-fold Δ (×1e-5) | folds Δ>0 |
|---|---:|---:|---:|---|---:|
| a route profile | 62 | 0.960333 | **+0.001244** | +118 +150 +129 +114 +112 | 5/5 |
| b value counts | 25 | 0.959810 | **+0.000721** | +84 +74 +67 +62 +71 | 5/5 |
| c fold-safe TE | 29 | 0.960425 | **+0.001336** | +148 +131 +129 +127 +133 | 5/5 |
| d original lookup / teacher | 43 | 0.960324 | **+0.001235** | +118 +129 +143 +120 +108 | 5/5 |
| e rating x context | 60 | 0.958609 | **-0.000479** | -74 -6 -57 -55 -49 | 0/5 |

| step | features | OOF AUC | ΔAUC vs full set | per-fold Δ (×1e-5) | folds Δ>0 |
|---|---:|---:|---:|---|---:|
| - a route profile | 107 | 0.960829 | **-0.000285** | -31 -31 -41 -25 -18 | 0/5 |
| - b value counts | 144 | 0.961052 | **-0.000062** | -22 -1 -4 +0 -13 | 1/5 |
| - c fold-safe TE | 140 | 0.960818 | **-0.000296** | -42 -1 -38 -28 -44 | 0/5 |
| - d original lookup / teacher | 126 | 0.960912 | **-0.000202** | -23 -2 -35 -20 -24 | 0/5 |
| - e rating x context | 109 | 0.961174 | **+0.000060** | -0 +12 -2 +12 +2 | 3/5 |

**How the ladder was measured.** LightGBM (`num_leaves=127, lr=0.03, feature_fraction=0.5, bagging 0.8, lambda_l2=5`),
early stopping on the validation fold (patience 300), all five folds. All ablations use the same folds and
early-stopping procedure. Selection optimism may differ between configurations; paired deltas improve comparability
but do not remove this bias. The embedded ablations were generated by the repository pipeline. This notebook implements
the final feature set; the rating-context crosses and the 33-column auxiliary variant are not included. Re-running the
20 ablations here would take hours, so the numbers are embedded (and written to `ladder_results.csv`).

Our contribution is a consolidated comparison of these established feature blocks on one shared split, plus reusable
prediction files for two final models. Fold deltas are descriptive, not confidence intervals.


In [ ]:
LADDER = {
 "stack": [
  {
   "label": "raw 21 columns",
   "blocks": "-",
   "n_feat": 21,
   "auc": 0.959089
  },
  {
   "label": "+ a route profile",
   "blocks": "a",
   "n_feat": 62,
   "auc": 0.960333,
   "delta": 0.001244,
   "fold_delta": [
    0.0011813,
    0.0014985,
    0.0012913,
    0.0011412,
    0.0011226
   ]
  },
  {
   "label": "+ b value counts",
   "blocks": "ab",
   "n_feat": 66,
   "auc": 0.960402,
   "delta": 6.9e-05,
   "fold_delta": [
    3.93e-05,
    0.0001255,
    -1.74e-05,
    0.0001714,
    -1.7e-05
   ]
  },
  {
   "label": "+ c fold-safe TE",
   "blocks": "abc",
   "n_feat": 74,
   "auc": 0.960464,
   "delta": 6.1e-05,
   "fold_delta": [
    0.0002083,
    5.9e-06,
    5.23e-05,
    -7.6e-06,
    0.0001059
   ]
  },
  {
   "label": "+ d original lookup / teacher",
   "blocks": "abcd",
   "n_feat": 96,
   "auc": 0.960941,
   "delta": 0.000478,
   "fold_delta": [
    0.0003694,
    0.0003975,
    0.000608,
    0.0005831,
    0.000458
   ]
  },
  {
   "label": "+ e rating x context crosses",
   "blocks": "abcde",
   "n_feat": 135,
   "auc": 0.96095,
   "delta": 9e-06,
   "fold_delta": [
    -0.0001193,
    4e-06,
    2.52e-05,
    4.96e-05,
    8.52e-05
   ]
  },
  {
   "label": "+ f aux-task (33 cols)",
   "blocks": "abcdef",
   "n_feat": 168,
   "auc": 0.961089,
   "delta": 0.000139,
   "fold_delta": [
    0.0002914,
    0.0001264,
    4.24e-05,
    0.0001037,
    6.52e-05
   ]
  },
  {
   "label": "f -> expected values only (13)",
   "blocks": "abcdef-ev",
   "n_feat": 148,
   "auc": 0.961114,
   "delta": 2.5e-05,
   "fold_delta": [
    1.18e-05,
    -1.5e-05,
    0.0001672,
    -4.2e-05,
    7.57e-05
   ]
  },
  {
   "label": "- e  (final set)",
   "blocks": "abcdf-ev",
   "n_feat": 109,
   "auc": 0.961174,
   "delta": 6e-05,
   "fold_delta": [
    -4.1e-06,
    0.0001239,
    -1.82e-05,
    0.0001208,
    2.14e-05
   ]
  }
 ],
 "single": [
  {
   "label": "a route profile",
   "blocks": "a",
   "n_feat": 62,
   "auc": 0.960333,
   "delta": 0.001244,
   "fold_delta": [
    0.0011813,
    0.0014985,
    0.0012913,
    0.0011412,
    0.0011226
   ]
  },
  {
   "label": "b value counts",
   "blocks": "b",
   "n_feat": 25,
   "auc": 0.95981,
   "delta": 0.000721,
   "fold_delta": [
    0.0008392,
    0.0007374,
    0.0006687,
    0.0006164,
    0.0007125
   ]
  },
  {
   "label": "c fold-safe TE",
   "blocks": "c",
   "n_feat": 29,
   "auc": 0.960425,
   "delta": 0.001336,
   "fold_delta": [
    0.0014761,
    0.0013051,
    0.0012919,
    0.0012691,
    0.001329
   ]
  },
  {
   "label": "d original lookup / teacher",
   "blocks": "d",
   "n_feat": 43,
   "auc": 0.960324,
   "delta": 0.001235,
   "fold_delta": [
    0.0011813,
    0.0012898,
    0.0014286,
    0.0011951,
    0.0010844
   ]
  },
  {
   "label": "e rating x context",
   "blocks": "e",
   "n_feat": 60,
   "auc": 0.958609,
   "delta": -0.000479,
   "fold_delta": [
    -0.0007385,
    -5.86e-05,
    -0.0005691,
    -0.0005505,
    -0.0004892
   ]
  }
 ],
 "drop": [
  {
   "label": "- a route profile",
   "blocks": "a",
   "n_feat": 107,
   "auc": 0.960829,
   "delta": -0.000285,
   "fold_delta": [
    -0.0003055,
    -0.0003061,
    -0.0004127,
    -0.0002483,
    -0.0001831
   ]
  },
  {
   "label": "- b value counts",
   "blocks": "b",
   "n_feat": 144,
   "auc": 0.961052,
   "delta": -6.2e-05,
   "fold_delta": [
    -0.0002158,
    -6.8e-06,
    -3.58e-05,
    3.1e-06,
    -0.0001348
   ]
  },
  {
   "label": "- c fold-safe TE",
   "blocks": "c",
   "n_feat": 140,
   "auc": 0.960818,
   "delta": -0.000296,
   "fold_delta": [
    -0.0004202,
    -1.44e-05,
    -0.0003752,
    -0.0002813,
    -0.0004448
   ]
  },
  {
   "label": "- d original lookup / teacher",
   "blocks": "d",
   "n_feat": 126,
   "auc": 0.960912,
   "delta": -0.000202,
   "fold_delta": [
    -0.0002315,
    -1.61e-05,
    -0.0003533,
    -0.0001965,
    -0.0002376
   ]
  },
  {
   "label": "- e rating x context",
   "blocks": "e",
   "n_feat": 109,
   "auc": 0.961174,
   "delta": 6e-05,
   "fold_delta": [
    -4.1e-06,
    0.0001239,
    -1.82e-05,
    0.0001208,
    2.14e-05
   ]
  }
 ]
}

VIEW = {"stack": "cum", "single": "alone", "drop": "drop"}
rows = [dict(r, view=VIEW[k]) for k in ["stack", "single", "drop"] for r in LADDER[k] if "delta" in r]
pd.DataFrame([{"view": r["view"], "step": r["label"], "blocks": r["blocks"], "n_features": r["n_feat"], "oof_auc": r["auc"],
               "delta_auc": r["delta"], **{f"fold{i}_delta": v for i, v in enumerate(r["fold_delta"])}} for r in rows]
             ).to_csv("ladder_results.csv", index=False)
BIG = 2e-4
groups = [("large effects  (|ΔAUC| ≥ 2e-4)", [r for r in rows if abs(r["delta"]) >= BIG]),
          ("small effects  (|ΔAUC| < 2e-4)", [r for r in rows if abs(r["delta"]) < BIG])]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for ax, (title, rr) in zip(axes, groups):
    pos = np.arange(len(rr))[::-1]
    d = np.array([r["delta"] for r in rr]) * 1e4
    ax.barh(pos, d, color=["#2a78d6" if v > 0 else "#eb6834" for v in d], height=0.6)
    for p, r in zip(pos, rr):
        fd = np.array(r["fold_delta"]) * 1e4
        ax.scatter(fd, np.full(5, p) + np.linspace(-0.18, 0.18, 5), s=14, color="#222", zorder=3)
    ax.axvline(0, color="#888", lw=1); ax.set_yticks(pos)
    ax.set_yticklabels([f"[{r['view']}] {r['label']}" for r in rr], fontsize=9)
    ax.set_title(title, fontsize=10, loc="left"); ax.set_xlabel("ΔAUC × 1e-4  (bar = pooled, dots = folds)")
    ax.grid(axis="x", alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("LightGBM feature ladder on the shared 5-fold split (blue = gain, orange = loss)", x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 4. Feature blocks

| block | labels used | columns | definition |
|---|---|---:|---|
| a route profile | none (train+test) | 41 | `fd_cnt`; for every other column, its mean within the Flight Distance value (`fdm_*`) and the row's residual from it (`res_*`) |
| b value counts | none (train+test) | 4 | occurrence count of each numeric value |
| c fold-safe TE | competition labels, training part of the outer fold only | 8 | keys FD, Age, FD×Age, FD×Class, FD×Travel, Age×Class×Travel×CustType, FD//10, FD//100; smoothing m=20; training rows cross-fitted with an inner 5-fold split, validation/test rows encoded with the full training-part statistics |
| d original lookup / teacher | original-dataset labels only | 22 | smoothed satisfaction rate of each value in the original 129,880 rows (`org_mean_*`, m=10) + logit of a LightGBM trained on the original rows only (`org_teacher`) |
| f aux-task expectations | none | 13 | for each rating, a 6-class XGBoost predicts it from the other 20 columns; `AUX_MODE='orig'` (default): one fit on the original rows, applied to all competition rows; `'full'`: 5-fold OOF over all train+test rows; feature = expected rating Σ k·P(k) |

For each inner split of block c, both the category statistics and the smoothing prior are computed from the inner
training rows only. Route profiles and counts use all competition features, including outer-validation and test rows.
In full mode, auxiliary models use a separate KFold split over train+test. No competition satisfaction labels are used;
this is transductive preprocessing, not a fully inductive pipeline. The lookup and teacher use only original-dataset
labels. We have not audited exact or near-duplicate overlap with the competition rows. Original rows are **not**
appended to train (public reports and our checks found that hurts).

In [ ]:
def block_route(df):
    g = df.groupby(FD)
    out = {"fd_cnt": g[FD].transform("size").astype(np.float32)}
    for c in FEATURES:
        if c != FD:
            m = g[c].transform("mean").astype(np.float32)
            out[f"fdm_{short(c)}"] = m
            out[f"res_{short(c)}"] = (df[c] - m).astype(np.float32)
    return pd.DataFrame(out)

def block_counts(df):
    return pd.DataFrame({f"cnt_{short(c)}": df[c].map(df[c].value_counts(dropna=False)).astype(np.float32) for c in NUMERIC})

def block_orig(df, m=10.0):
    prior, out = y_o.mean(), {}
    for c in FEATURES:
        s = pd.DataFrame({"v": X_o[c].to_numpy(), "y": y_o}).groupby("v", dropna=False)["y"].agg(["sum", "count"])
        out[f"org_mean_{short(c)}"] = df[c].map((s["sum"] + m * prior) / (s["count"] + m)).fillna(prior).astype(np.float32)
    params = dict(objective="binary", learning_rate=0.05, max_depth=8, num_leaves=127, feature_fraction=0.5,
                  lambda_l2=5.0, min_child_samples=20, n_jobs=NCPU, seed=42, verbose=-1)
    teacher = lgb.train(params, lgb.Dataset(X_o, y_o, categorical_feature=CATEGORICAL), 600)
    p = np.clip(teacher.predict(df[FEATURES]), 1e-6, 1 - 1e-6)
    out["org_teacher"] = np.log(p / (1 - p)).astype(np.float32)
    return pd.DataFrame(out)

def block_aux_ev(df, mode, rounds=AUX_ROUNDS, seed=42):
    """Expected rating E[rating | other 20 columns] from a 6-class XGBoost per rating.
    mode "orig": one fit per rating on the 129,880 original rows, applied to all competition rows (13 fits, CPU-friendly).
    mode "full": 5-fold OOF over all train+test rows (as in our ladder; 65 fits, needs a GPU to be quick)."""
    allX, out = df.astype(np.float32), {}
    srcX = X_o.astype(np.float32) if mode == "orig" else allX
    for col in RATINGS:
        t0 = time.time()
        vals = np.sort(srcX[col].dropna().unique()); yk = np.searchsorted(vals, srcX[col].to_numpy())
        params = dict(objective="multi:softprob", num_class=len(vals), eval_metric="mlogloss", tree_method="hist",
                      device="cuda" if GPU else "cpu", nthread=NCPU, learning_rate=0.1, max_depth=7, min_child_weight=5,
                      subsample=0.8, colsample_bytree=0.8, max_bin=256, seed=seed)
        X = allX.drop(columns=[col]).to_numpy()
        if mode == "orig":
            m = xgb.train(params, xgb.QuantileDMatrix(srcX.drop(columns=[col]).to_numpy(), yk), rounds)
            P = m.predict(xgb.DMatrix(X)); del m
        else:
            P = np.zeros((len(df), len(vals)), np.float32)
            for tr, va in KFold(5, shuffle=True, random_state=seed).split(df):
                m = xgb.train(params, xgb.QuantileDMatrix(X[tr], yk[tr]), rounds)
                P[va] = m.predict(xgb.DMatrix(X[va])); del m
        out[f"aux_ev_{short(col)}"] = (P * vals.astype(np.float32)).sum(1)
        print(f"  aux {col}: {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(out)

TE_KEYS = {"fd": [FD], "age": ["Age"], "fd_age": [FD, "Age"], "fd_class": [FD, "Class"], "fd_tot": [FD, "Type of Travel"],
           "age_cls_tot_ct": ["Age", "Class", "Type of Travel", "Customer Type"], "fd10": ["fd10"], "fd100": ["fd100"]}

def te_keys(df):
    b = df.copy(); b["fd10"] = b[FD] // 10; b["fd100"] = b[FD] // 100
    return pd.DataFrame({k: pd.MultiIndex.from_frame(b[c]).factorize()[0] if len(c) > 1 else pd.factorize(b[c[0]])[0]
                         for k, c in TE_KEYS.items()})

def target_encode(keys, y_tr, idx_tr, idx_other, m=20.0, inner=5, seed=42):
    """Training rows: inner 5-fold cross-fit; for each inner split, both the category statistics and the smoothing
    prior come from the inner training rows only. Other rows (outer validation, test): full training-part statistics."""
    inner_split = list(StratifiedKFold(inner, shuffle=True, random_state=seed).split(idx_tr, y_tr))
    def table(k, yy, n):
        return np.bincount(k, weights=yy, minlength=n), np.bincount(k, minlength=n).astype(np.float64)
    def look(s, c, k, prior):
        return ((s[k] + m * prior) / (c[k] + m)).astype(np.float32)
    tr_out, ot_out = {}, {}
    for name in keys.columns:
        k = keys[name].to_numpy(); n = int(k.max()) + 1; ktr = k[idx_tr]
        enc = np.zeros(len(idx_tr), np.float32)
        for a, b in inner_split:
            enc[b] = look(*table(ktr[a], y_tr[a], n), ktr[b], float(y_tr[a].mean()))
        tr_out[f"te_{name}"] = enc
        ot_out[f"te_{name}"] = look(*table(ktr, y_tr, n), k[idx_other], float(y_tr.mean()))
    return pd.DataFrame(tr_out), pd.DataFrame(ot_out)

t0 = time.time()
raw = both.astype(np.float32)
for c in CATEGORICAL:
    v = raw[c].astype(np.int64); raw[c] = pd.Categorical(v, categories=np.sort(v.unique()))
blocks = {"raw": raw, "a": block_route(both), "b": block_counts(both)}
print(f"a+b built ({time.time() - t0:.0f}s)"); t0 = time.time()
blocks["d"] = block_orig(both); print(f"d built ({time.time() - t0:.0f}s)"); t0 = time.time()
blocks["f"] = block_aux_ev(both, AUX_MODE); T_AUX = time.time() - t0
print(f"f built ({T_AUX:.0f}s, mode {AUX_MODE}, {'GPU' if GPU else 'CPU'})")
FRAME = pd.concat([blocks[k] for k in ["raw", "a", "b", "d", "f"]], axis=1)
KEYS = te_keys(both)
print("static frame", FRAME.shape, "+ 8 TE columns per fold")

## 5. Final single models (fixed-round refit OOF)

Parameters: LightGBM `num_leaves=127, lr=0.015, feature_fraction=0.5, bagging 0.8, lambda_l2=5, min_child_samples=50`,
XGBoost `depth=8, lr=0.01, colsample=0.5, subsample=0.8, min_child_weight=5, max_cat_to_onehot=1`.

These are fixed-round refit OOF predictions. The round counts were selected using early stopping on the same five
folds, so the reported AUC is not an independent estimate and may retain selection optimism. The counts (1084 / 1438)
are the mean best iteration of that early-stopped pass; here every fold is trained for exactly that many rounds. Features
and parameters were also chosen on the same CV. Set `RUN_PASS1 = True` to re-derive the counts (doubles the runtime).

Reference values: the recorded Kaggle CPU run (`AUX_MODE='orig'`) gave LightGBM 0.961103 / XGBoost
0.961150; repository runs with the full aux block gave 0.961198 / 0.961197. The
recorded run predates the per-inner-split TE prior of block c, and library versions and thread counts can move the OOF
by a few 1e-5, so this version's printed OOF can differ slightly.

In [ ]:
def fold_frames(f):
    tr, va = np.where(FOLDS != f)[0], np.where(FOLDS == f)[0]
    te = np.arange(n_train, len(FRAME))
    te_tr, te_ot = target_encode(KEYS, y[tr], tr, np.concatenate([va, te]))
    Xtr = pd.concat([FRAME.iloc[tr].reset_index(drop=True), te_tr], axis=1)
    Xva = pd.concat([FRAME.iloc[va].reset_index(drop=True), te_ot.iloc[:len(va)].reset_index(drop=True)], axis=1)
    Xte = pd.concat([FRAME.iloc[te].reset_index(drop=True), te_ot.iloc[len(va):].reset_index(drop=True)], axis=1)
    return tr, va, Xtr, Xva, Xte

LGB_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.015, num_leaves=127, min_child_samples=50,
                  feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, max_cat_to_onehot=8,
                  cat_smooth=10, cat_l2=10, max_bin=256, n_jobs=NCPU, seed=42, verbose=-1)
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist", device="cuda" if GPU else "cpu",
                  nthread=NCPU, learning_rate=0.01, max_depth=8, min_child_weight=5, subsample=0.8, colsample_bytree=0.5,
                  reg_lambda=1.0, max_cat_to_onehot=1, max_bin=256, seed=42)

def fit(model, Xtr, ytr, Xva, yva, Xte, rounds, early_stop=False):
    if model == "lgb":
        dtr = lgb.Dataset(Xtr, ytr)
        if early_stop:
            m = lgb.train(LGB_PARAMS, dtr, 20000, valid_sets=[lgb.Dataset(Xva, yva, reference=dtr)],
                          callbacks=[lgb.early_stopping(300, verbose=False)])
            return None, None, m.best_iteration
        m = lgb.train(LGB_PARAMS, dtr, rounds)
        return m.predict(Xva), m.predict(Xte), rounds
    dtr = xgb.DMatrix(Xtr, ytr, enable_categorical=True)
    dva = xgb.DMatrix(Xva, yva, enable_categorical=True)
    if early_stop:
        m = xgb.train(XGB_PARAMS, dtr, 20000, evals=[(dva, "va")], early_stopping_rounds=300, verbose_eval=False)
        return None, None, m.best_iteration + 1
    m = xgb.train(XGB_PARAMS, dtr, rounds)
    return m.predict(dva), m.predict(xgb.DMatrix(Xte, enable_categorical=True)), rounds

def run(model, rounds):
    t_model = time.time()
    if RUN_PASS1:     # pass 1: early stopping on each validation fold, only to choose the round count
        its = [fit(model, Xtr, y[tr], Xva, y[va], Xte, None, True)[2] for tr, va, Xtr, Xva, Xte in map(fold_frames, range(5))]
        rounds = int(round(np.mean(its))); print(f"{model} pass 1 best iterations {its} -> {rounds}")
    oof, pred = np.zeros(n_train), np.zeros(len(FRAME) - n_train)
    for f in range(5):
        t0 = time.time()
        tr, va, Xtr, Xva, Xte = fold_frames(f)
        oof[va], p_te, _ = fit(model, Xtr, y[tr], Xva, y[va], Xte, rounds)
        pred += p_te / 5
        print(f"{model} fold {f}: AUC {roc_auc_score(y[va], oof[va]):.6f}  ({Xtr.shape[1]} features, {rounds} rounds, {time.time() - t0:.0f}s)", flush=True)
    print(f"{model} fixed-round refit OOF AUC {roc_auc_score(y, oof):.6f}  ({(time.time() - t_model) / 60:.1f} min)")
    return oof, pred, rounds, Xtr.shape[1], time.time() - t_model

RES = {"lgb": run("lgb", LGB_ROUNDS)}
if RUN_XGB:
    RES["xgb"] = run("xgb", XGB_ROUNDS)

## 6. Outputs: submission + stack-ready OOF

`submission.csv` is the single LightGBM. The default Kaggle CPU run produced LightGBM OOF AUC 0.961103 and
public LB 0.96098. `submission_xgb.csv` is the single XGBoost; no public LB is reported here for it. The
OOF/test files hold raw probabilities on the shared `StratifiedKFold(5, shuffle=True, random_state=42)` split, keyed by
`id`, with the `fold` column. Join predictions by id and verify the per-id fold assignments before stacking. Matching
splitter settings alone does not guarantee identical folds. These files retain the model-selection caveat described
above. `run_metadata.json` records the aux mode, round counts, library versions and an id/fold hash; `ladder_results.csv`
holds the embedded ladder. The logit average of the two GBDTs is shown only for reference.

```python
oof = pd.read_csv("/kaggle/input/s6e10-feature-ladder-what-helps-lightgbm/oof_feature_ladder.csv")   # after adding this notebook's output as input
mine = mine.merge(oof[["id", "fold", "oof_lgb", "oof_xgb"]], on="id", validate="1:1", suffixes=("", "_fl"))
assert (mine["fold"] == mine["fold_fl"]).all(), "different folds: do not stack these OOFs"
```

In [ ]:
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
oof_df = pd.DataFrame({"id": train[ID].to_numpy(), "fold": FOLDS, TARGET: y, **{f"oof_{k}": v[0] for k, v in RES.items()}})
test_df = pd.DataFrame({"id": test[ID].to_numpy(), **{f"pred_{k}": v[1] for k, v in RES.items()}})
oof_df.to_csv("oof_feature_ladder.csv", index=False); test_df.to_csv("test_feature_ladder.csv", index=False)
sub = pd.DataFrame({"id": test[ID].to_numpy(), TARGET: RES["lgb"][1]})
sub.to_csv("submission.csv", index=False)
if "xgb" in RES:
    pd.DataFrame({"id": test[ID].to_numpy(), TARGET: RES["xgb"][1]}).to_csv("submission_xgb.csv", index=False)
    from scipy.stats import spearmanr
    blend = (logit(RES["lgb"][0]) + logit(RES["xgb"][0])) / 2
    print(f"LGB {roc_auc_score(y, RES['lgb'][0]):.6f} | XGB {roc_auc_score(y, RES['xgb'][0]):.6f} | "
          f"logit mean {roc_auc_score(y, blend):.6f} | Spearman {spearmanr(RES['lgb'][0], RES['xgb'][0])[0]:.5f}")
assert sub.shape == (len(test), 2) and sub[TARGET].between(0, 1).all() and sub["id"].is_unique
META = {"debug": DEBUG, "aux_mode": AUX_MODE, "gpu": GPU, "n_cpu": NCPU, "aux_rounds": AUX_ROUNDS, "run_pass1": RUN_PASS1,
        "folds": "StratifiedKFold(5, shuffle=True, random_state=42) on train row order", "id_fold_sha256_16": FOLD_HASH,
        "inputs": {"train": TRAIN, "test": TEST, "original": ORIG},
        "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                     "sklearn": sklearn.__version__, "lightgbm": lgb.__version__, "xgboost": xgb.__version__},
        "models": {k: {"rounds": v[2], "n_features": v[3], "oof_auc": round(float(roc_auc_score(y, v[0])), 6),
                       "minutes": round(v[4] / 60, 1)} for k, v in RES.items()},
        "aux_minutes": round(T_AUX / 60, 1), "total_minutes": round((time.time() - T_START) / 60, 1)}
Path("run_metadata.json").write_text(json.dumps(META, indent=1))
print(sub.head(3)); print(json.dumps(META["models"])); print(f"total runtime {(time.time() - T_START) / 60:.1f} min")

## 7. Notes and credits

* **Ideas** (we re-implemented and measured them; the ideas are theirs):
  * [@goodpjw2008](https://www.kaggle.com/goodpjw2008) — Flight Distance as a route key and the TE key set
    ([Part 1](https://www.kaggle.com/code/goodpjw2008/s6e10-route-id-fe-og-model-stack-lb-0-96108)). Route-profile means
    and the auxiliary-feature implementation follow goodpjw2008
    ([Part 4](https://www.kaggle.com/code/goodpjw2008/s6e10-auxiliary-task-features-lb-0-96129)).
  * [@evgendvorkin](https://www.kaggle.com/code/evgendvorkin/s6e10-lightgbm-cv-5-folds-0-96111) — per-value original-data
    target means, original-only model probability as a feature, coarse distance keys.
  * Residual features are also present in evgendvorkin's notebook and are analyzed in
    [@busyaprime](https://www.kaggle.com/code/busyaprime/s6e10-route-or-distance-both-lb-0-96169)'s route decomposition
    (which also reports that appending original rows hurts, with paired-bootstrap comparisons).
  * [@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) — count encodings and combination target
    encodings in the RealMLP baseline.
  * Chris Deotte's [public-OOF stacking notebook](https://www.kaggle.com/code/cdeotte/stacking-tfms-with-public-oof)
    motivated our output format and compatibility checks; we use the same documented seed-42 split.
* **Data**: We use the processed dataset published by
  [arseniyshutko](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k) (CC0-1.0). The upstream
  [Airline Passenger Satisfaction](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction) dataset was
  published by TJ Klein, whose data card credits an earlier dataset by John D. Dataset terms and source-code licenses are
  separate; see the linked source pages.
* **Caveats**: ladder deltas of a few 1e-5 (b, c on top of a, e) are small relative to the fold-to-fold spread; read the
  per-fold dots. Small public-LB differences may be dominated by sampling variability; uncertainty depends on how
  similar the predictions are. The ladder is a LightGBM ablation; public reports suggest rating×context crosses can help
  CatBoost, which we did not re-test here. The GPU/full-aux path has not been validated in the recorded Kaggle run.
